# MeChess 2/2: train the chess-text language model (GPU notebook)

Trains one model that reads a comment or paragraph and predicts **which chess concepts it discusses** (with the keywords hidden, so it has to use context), **the human verdict on the move**
(`?? ? ?! !? ! !!`) and **who stands better** (`= ⩲ ± +− ...`). See `plan.md` section 8p and `chessme/books/nlp.py`.

**Before you spend GPU hours**
1. Run *MeChess 1/2* first (CPU) and add its output to this notebook: *Add Input -> Notebook output files*.
2. *Settings -> Accelerator -> GPU (T4 x2 or P100)*, *Internet -> On*. Put your repository URL in `REPO_URL`.
3. Click **Save Version -> Save & Run All (Commit)**. Cells 3 to 5 are the protection: preflight (dependencies, GPU, model download), then a **dry-run** of the whole training path on 400 examples
   (about a minute), and only then the real run. If any of them fails, the notebook stops within minutes, not hours.
4. The real run has a **time budget** (`BUDGET_MIN`, default 600 min = 10 h, under Kaggle's 12 h limit): at the deadline it saves a checkpoint and finishes normally, so the output is kept.
   To continue later, add this notebook's own output as an input and rerun: training resumes from the checkpoint.

In [ ]:
import os, subprocess, sys, pathlib, time, glob

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
MODEL = "distilroberta-base"                            # any Hugging Face encoder; a fast trial: "google/bert_uncased_L-2_H-128_A-2"; avoid DeBERTa unless you also install sentencepiece
EPOCHS = 3
BATCH = 64
BUDGET_MIN = 600                                        # wall-clock budget for the real run, in minutes
OUT = "/kaggle/working/learn" if os.path.exists("/kaggle") else "learn_out"
T0 = time.time()

def sh(*args):
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests")
CLI = [sys.executable, "-m", "chessme"]

# find the data written by notebook 1 (an input) and the checkpoint of an earlier run of this notebook (also an input, optional)
found = sorted(glob.glob("/kaggle/input/**/annotated_moves.jsonl.gz", recursive=True))
assert found, "add the output of notebook 1 as an input (Add Input -> Notebook output files)"
DATA = str(pathlib.Path(found[0]).parent.parent)
os.makedirs(OUT, exist_ok=True)
for name in ("annotated", "prose", "books"):                       # the training reads <data>/annotated, <data>/prose, <data>/books
    src, dst = pathlib.Path(DATA, name), pathlib.Path(OUT, name)
    if src.exists() and not dst.exists():
        os.symlink(src, dst)
prev = sorted(glob.glob("/kaggle/input/**/nlp/ckpt.pt", recursive=True))
if prev:
    os.makedirs(f"{OUT}/nlp", exist_ok=True)
    for f in glob.glob(str(pathlib.Path(prev[0]).parent / "*")):
        subprocess.run(["cp", f, f"{OUT}/nlp/"], check=True)
    print("resuming from", prev[0])
print("data:", DATA)

## Step 1. Preflight: GPU present, dependencies, the model can be downloaded

In [ ]:
sh(*CLI, "books-preflight", "--out", OUT, "--need-gpu", "--backend", "transformer", "--model", MODEL, "--no-network")

## Step 2. Dry-run: the whole training path on 400 examples (about a minute). If this fails, nothing expensive was spent.

In [ ]:
sh(*CLI, "books-nlp-train", "--data", OUT, "--out", f"{OUT}/nlp_dry", "--backend", "transformer", "--model", MODEL, "--dry-run", "--log", f"{OUT}/nlp_dry.log")

## Step 3. The real run, with a time budget (saves a checkpoint at the deadline and at regular intervals)

In [ ]:
remaining = max(5, BUDGET_MIN - (time.time() - T0) / 60)
sh(*CLI, "books-nlp-train", "--data", OUT, "--out", f"{OUT}/nlp", "--backend", "transformer", "--model", MODEL, "--epochs", str(EPOCHS), "--batch", str(BATCH),
   "--deadline-minutes", str(round(remaining, 2)), "--ckpt-every", "500", "--log", f"{OUT}/nlp.log")

## Step 4. Results and a few predictions

In [ ]:
import json
m = json.load(open(f"{OUT}/nlp/metrics.json"))
t = m["test"]
print(f"steps {m['steps']}/{m['total_steps']}  stopped early: {m['stopped']}  device {m['device']}  {m['minutes']} min")
for k in ("concept_f1_micro", "concept_f1_micro_prior", "concept_f1_macro", "judgement_f1_macro", "judgement_acc", "judgement_acc_majority", "eval_acc", "eval_acc_majority"):
    if k in t: print(f"  {k:26s} {t[k]:.3f}")
print("with keywords visible (test, unmasked): concept micro-F1", round(m["test_unmasked"]["concept_f1_micro"], 3))
sh(*CLI, "books-nlp-predict", f"{OUT}/nlp",
   "The knight can never be dislodged from d5 because no enemy pawn can attack that square.",
   "A terrible move, it loses a piece for nothing.",
   "Black is now completely lost.")

In [ ]:
# keep only what is needed: the trained model, its metrics and the logs (not the linked data)
import shutil
for name in ("annotated", "prose", "books", "nlp_dry"):
    p = pathlib.Path(OUT, name)
    if p.is_symlink(): p.unlink()
    elif p.is_dir(): shutil.rmtree(p, ignore_errors=True)
print("output kept:", sorted(os.listdir(OUT)))